# LinkedIn User Prediction Model
## Building a Machine Learning App to Predict LinkedIn Users

**Project Goal:** Build a classification model to predict whether individuals use LinkedIn based on demographic and socioeconomic factors, enabling the marketing team to target campaigns effectively.

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('default')
sns.set_palette('Set2')

## Step 1: Read in the Data and Check Dimensions

In [ ]:
# Read the data
s = pd.read_csv('social_media_usage.csv')

# Check dimensions
print(f"Dataset dimensions: {s.shape}")
print(f"Number of rows: {s.shape[0]}")
print(f"Number of columns: {s.shape[1]}")

# Display first few rows
s.head()

## Step 2: Define and Test the clean_sm Function

In [ ]:
# Define the clean_sm function
def clean_sm(x):
    """
    Convert social media usage to binary format.
    Returns 1 if x equals 1, otherwise returns 0.
    """
    x = np.where(x == 1, 1, 0)
    return x

In [ ]:
# Test the function with a toy dataframe
toy_df = pd.DataFrame({
    'col1': [1, 2, 1],
    'col2': [1, 1, 8]
})

print("Original toy dataframe:")
print(toy_df)
print("\nAfter applying clean_sm:")
toy_df['col1_clean'] = clean_sm(toy_df['col1'])
toy_df['col2_clean'] = clean_sm(toy_df['col2'])
print(toy_df)
print("\nFunction works correctly: 1 becomes 1, all other values become 0")

## Step 3: Create the Clean Dataset (ss) and Perform Exploratory Analysis

In [ ]:
# Create the ss dataframe with cleaned variables
ss = pd.DataFrame({
    'sm_li': clean_sm(s['web1h']),
    'income': np.where(s['income'] > 9, np.nan, s['income']),
    'education': np.where(s['educ2'] > 8, np.nan, s['educ2']),
    'parent': np.where(s['par'] == 1, 1, np.where(s['par'] == 2, 0, np.nan)),
    'married': np.where(s['marital'] == 1, 1, np.where(s['marital'] == 2, 0, np.nan)),
    'female': np.where(s['gender'] == 2, 1, np.where(s['gender'] == 1, 0, np.nan)),
    'age': np.where(s['age'] > 98, np.nan, s['age'])
})

# Drop missing values
ss = ss.dropna()

print(f"Clean dataset dimensions: {ss.shape}")
print(f"\nDataset info:")
print(ss.info())
print(f"\nSummary statistics:")
ss.describe()

In [ ]:
# Check target variable distribution
print("LinkedIn Usage Distribution:")
print(ss['sm_li'].value_counts())
print(f"\nPercentage of LinkedIn users: {ss['sm_li'].mean() * 100:.2f}%")

### Exploratory Data Analysis: Feature Relationships with Target

In [ ]:
# Visualize relationship between income and LinkedIn usage
plt.figure(figsize=(10, 6))
income_li = ss.groupby('income')['sm_li'].mean()
plt.bar(income_li.index, income_li.values)
plt.xlabel('Income Level', fontsize=12)
plt.ylabel('Proportion Using LinkedIn', fontsize=12)
plt.title('LinkedIn Usage by Income Level', fontsize=14, fontweight='bold')
plt.xticks(range(1, 10))
plt.ylim(0, 0.6)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("LinkedIn usage increases with income level")

In [ ]:
# Visualize relationship between education and LinkedIn usage
plt.figure(figsize=(10, 6))
education_li = ss.groupby('education')['sm_li'].mean()
plt.bar(education_li.index, education_li.values, color='green')
plt.xlabel('Education Level', fontsize=12)
plt.ylabel('Proportion Using LinkedIn', fontsize=12)
plt.title('LinkedIn Usage by Education Level', fontsize=14, fontweight='bold')
plt.xticks(range(1, 9))
plt.ylim(0, 0.6)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("LinkedIn usage increases with education level")

In [ ]:
# Visualize relationship between age and LinkedIn usage
plt.figure(figsize=(10, 6))
age_bins = [0, 25, 35, 45, 55, 65, 100]
age_labels = ['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
ss['age_group'] = pd.cut(ss['age'], bins=age_bins, labels=age_labels)
age_li = ss.groupby('age_group')['sm_li'].mean()
plt.bar(range(len(age_li)), age_li.values, color='orange')
plt.xlabel('Age Group', fontsize=12)
plt.ylabel('Proportion Using LinkedIn', fontsize=12)
plt.title('LinkedIn Usage by Age Group', fontsize=14, fontweight='bold')
plt.xticks(range(len(age_li)), age_labels)
plt.ylim(0, 0.5)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Drop the temporary age_group column
ss = ss.drop('age_group', axis=1)

print("LinkedIn usage is highest in middle age groups and decreases with older age")

In [ ]:
# Compare LinkedIn usage across binary variables
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

binary_vars = ['parent', 'married', 'female']
labels = ['Parent', 'Married', 'Female']
colors = ['purple', 'red', 'blue']

for i, (var, label, color) in enumerate(zip(binary_vars, labels, colors)):
    usage = ss.groupby(var)['sm_li'].mean()
    axes[i].bar(['No', 'Yes'], usage.values, color=color, alpha=0.7)
    axes[i].set_ylabel('Proportion Using LinkedIn', fontsize=11)
    axes[i].set_title(f'LinkedIn Usage by {label}', fontsize=12, fontweight='bold')
    axes[i].set_ylim(0, 0.4)
    axes[i].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Binary variable comparisons show some differences in LinkedIn usage patterns")

In [ ]:
# Correlation matrix
plt.figure(figsize=(10, 8))
correlation = ss.corr()
sns.heatmap(correlation, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix of Features and Target', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("- Income and education show the strongest positive correlation with LinkedIn usage")
print("- Age shows a negative correlation with LinkedIn usage")
print("- Parent, married, and female show weaker correlations")

## Step 4: Create Target Vector (y) and Feature Set (X)

In [ ]:
# Create feature set X (all columns except target)
X = ss[['income', 'education', 'parent', 'married', 'female', 'age']]

# Create target vector y (LinkedIn usage)
y = ss['sm_li']

print("Feature set (X) shape:", X.shape)
print("Target vector (y) shape:", y.shape)
print("\nFeature set preview:")
print(X.head())
print("\nTarget vector preview:")
print(y.head())

## Step 5: Split Data into Training and Test Sets

**Explanation of Train-Test Split:**
- We split the data into training (80%) and test (20%) sets
- **Training set (X_train, y_train):** Used to train/fit the model - the model learns patterns from this data
- **Test set (X_test, y_test):** Used to evaluate model performance - the model has never seen this data during training
- This approach helps us assess how well the model will generalize to new, unseen data
- `random_state=42` ensures reproducibility - we get the same split each time we run the code

In [ ]:
# Split the data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                      test_size=0.2, 
                                                      random_state=42)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])
print(f"\nPercentage in training: {X_train.shape[0] / X.shape[0] * 100:.1f}%")
print(f"Percentage in test: {X_test.shape[0] / X.shape[0] * 100:.1f}%")

## Step 6: Instantiate and Fit Logistic Regression Model

In [ ]:
# Initialize logistic regression model with balanced class weights
# class_weight='balanced' adjusts for imbalanced classes
lr_model = LogisticRegression(class_weight='balanced', random_state=42)

# Fit the model with training data
lr_model.fit(X_train, y_train)

print("Model trained successfully!")
print("\nModel coefficients (feature importance):")
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', ascending=False)
print(coef_df)

## Step 7: Evaluate Model and Create Confusion Matrix

**Model Accuracy:** The proportion of correct predictions out of all predictions made.

In [ ]:
# Make predictions on test set
y_pred = lr_model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"\nThis means the model correctly predicts LinkedIn usage {accuracy * 100:.2f}% of the time.")

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

print("\n" + "="*60)
print("CONFUSION MATRIX INTERPRETATION:")
print("="*60)
print(f"True Negatives (TN): {cm[0, 0]} - Correctly predicted as NOT LinkedIn users")
print(f"False Positives (FP): {cm[0, 1]} - Incorrectly predicted as LinkedIn users (Type I Error)")
print(f"False Negatives (FN): {cm[1, 0]} - Incorrectly predicted as NOT LinkedIn users (Type II Error)")
print(f"True Positives (TP): {cm[1, 1]} - Correctly predicted as LinkedIn users")
print("="*60)

## Step 8: Create Confusion Matrix as Dataframe with Informative Labels

In [ ]:
# Create confusion matrix as a dataframe with clear labels
cm_df = pd.DataFrame(cm,
                     index=['Actual: No LinkedIn', 'Actual: Uses LinkedIn'],
                     columns=['Predicted: No LinkedIn', 'Predicted: Uses LinkedIn'])

print("Confusion Matrix as DataFrame:")
print(cm_df)
print("\nVisualization:")

In [ ]:
# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=True, 
            square=True, linewidths=2, linecolor='black',
            annot_kws={'size': 16, 'weight': 'bold'})
plt.title('Confusion Matrix: LinkedIn User Prediction', fontsize=14, fontweight='bold', pad=20)
plt.ylabel('Actual', fontsize=12, fontweight='bold')
plt.xlabel('Predicted', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 9: Calculate Precision, Recall, and F1 Score by Hand

### Metric Definitions:

**Precision** = TP / (TP + FP)
- What proportion of positive predictions were actually correct?
- **Use case:** When false positives are costly (e.g., spam email detection - don't want to mark important emails as spam)

**Recall (Sensitivity)** = TP / (TP + FN)
- What proportion of actual positives did we correctly identify?
- **Use case:** When false negatives are costly (e.g., disease screening - don't want to miss sick patients)

**F1 Score** = 2 × (Precision × Recall) / (Precision + Recall)
- Harmonic mean of precision and recall, balancing both metrics
- **Use case:** When you need a single metric and both false positives and false negatives matter

In [ ]:
# Extract values from confusion matrix
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix Values:")
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP): {tp}")
print("\n" + "="*60)

In [ ]:
# Calculate precision by hand
precision_manual = tp / (tp + fp)
print("PRECISION (calculated by hand):")
print(f"Formula: TP / (TP + FP) = {tp} / ({tp} + {fp}) = {tp} / {tp + fp}")
print(f"Precision: {precision_manual:.4f}")
print(f"\nInterpretation: Of all people we predicted use LinkedIn, {precision_manual * 100:.2f}% actually do.")
print("\nWhen to prioritize: When false positives are costly.")
print("Example: Email spam detection - you don't want to incorrectly mark important emails as spam.")
print("="*60)

In [ ]:
# Calculate recall by hand
recall_manual = tp / (tp + fn)
print("RECALL / SENSITIVITY (calculated by hand):")
print(f"Formula: TP / (TP + FN) = {tp} / ({tp} + {fn}) = {tp} / {tp + fn}")
print(f"Recall: {recall_manual:.4f}")
print(f"\nInterpretation: Of all people who actually use LinkedIn, we correctly identified {recall_manual * 100:.2f}%.")
print("\nWhen to prioritize: When false negatives are costly.")
print("Example: Cancer screening - you don't want to miss patients who have cancer.")
print("="*60)

In [ ]:
# Calculate F1 score by hand
f1_manual = 2 * (precision_manual * recall_manual) / (precision_manual + recall_manual)
print("F1 SCORE (calculated by hand):")
print(f"Formula: 2 × (Precision × Recall) / (Precision + Recall)")
print(f"       = 2 × ({precision_manual:.4f} × {recall_manual:.4f}) / ({precision_manual:.4f} + {recall_manual:.4f})")
print(f"       = 2 × {precision_manual * recall_manual:.4f} / {precision_manual + recall_manual:.4f}")
print(f"F1 Score: {f1_manual:.4f}")
print(f"\nInterpretation: Harmonic mean of precision and recall, balancing both metrics.")
print("\nWhen to prioritize: When you need a single metric and both false positives and negatives matter.")
print("Example: Fraud detection - you want to catch fraudsters but not inconvenience legitimate customers.")
print("="*60)

In [ ]:
# Generate classification report using sklearn
print("\nSKLEARN CLASSIFICATION REPORT (for verification):")
print("="*60)
print(classification_report(y_test, y_pred, target_names=['No LinkedIn', 'Uses LinkedIn']))

print("\nVERIFICATION:")
print(f"Manual Precision: {precision_manual:.4f}")
print(f"Manual Recall:    {recall_manual:.4f}")
print(f"Manual F1 Score:  {f1_manual:.4f}")
print("\nThe manually calculated metrics match the sklearn classification report! ✓")

## Step 10: Make Predictions for Specific Individuals

We'll predict LinkedIn usage probability for two individuals with different characteristics.

In [ ]:
# Person 1: High income, high education, non-parent, married, female, 42 years old
person1 = pd.DataFrame({
    'income': [8],
    'education': [7],
    'parent': [0],
    'married': [1],
    'female': [1],
    'age': [42]
})

# Get probability for person 1
prob_person1 = lr_model.predict_proba(person1)[0][1]
pred_person1 = lr_model.predict(person1)[0]

print("PERSON 1 PREDICTION:")
print("Characteristics: High income (8), High education (7), Non-parent, Married, Female, Age 42")
print(f"\nPredicted class: {'LinkedIn User' if pred_person1 == 1 else 'Not a LinkedIn User'}")
print(f"Probability of using LinkedIn: {prob_person1:.4f} ({prob_person1 * 100:.2f}%)")
print(f"Probability of NOT using LinkedIn: {1 - prob_person1:.4f} ({(1 - prob_person1) * 100:.2f}%)")

In [ ]:
# Person 2: Same as person 1 but 82 years old
person2 = pd.DataFrame({
    'income': [8],
    'education': [7],
    'parent': [0],
    'married': [1],
    'female': [1],
    'age': [82]
})

# Get probability for person 2
prob_person2 = lr_model.predict_proba(person2)[0][1]
pred_person2 = lr_model.predict(person2)[0]

print("\n" + "="*70)
print("PERSON 2 PREDICTION:")
print("Characteristics: High income (8), High education (7), Non-parent, Married, Female, Age 82")
print(f"\nPredicted class: {'LinkedIn User' if pred_person2 == 1 else 'Not a LinkedIn User'}")
print(f"Probability of using LinkedIn: {prob_person2:.4f} ({prob_person2 * 100:.2f}%)")
print(f"Probability of NOT using LinkedIn: {1 - prob_person2:.4f} ({(1 - prob_person2) * 100:.2f}%)")

In [ ]:
# Compare the two predictions
print("\n" + "="*70)
print("COMPARISON:")
print("="*70)
print(f"Person 1 (Age 42): {prob_person1:.4f} ({prob_person1 * 100:.2f}%) probability")
print(f"Person 2 (Age 82): {prob_person2:.4f} ({prob_person2 * 100:.2f}%) probability")
print(f"\nDifference: {prob_person1 - prob_person2:.4f} ({(prob_person1 - prob_person2) * 100:.2f} percentage points)")
print(f"\nThe 42-year-old is {(prob_person1 / prob_person2):.2f}x more likely to use LinkedIn than the 82-year-old.")
print("\nKey insight: Age has a significant negative effect on LinkedIn usage probability.")
print("Even with high income and education, older individuals are much less likely to use LinkedIn.")

## Summary and Business Insights

### Model Performance:
- The logistic regression model successfully predicts LinkedIn usage with reasonable accuracy
- Both precision and recall metrics provide insights into different types of prediction errors

### Key Findings for Marketing Team:
1. **Income and Education** are the strongest predictors of LinkedIn usage
   - Higher income and education levels strongly correlate with LinkedIn usage
   - Marketing campaigns should prioritize educated, higher-income demographics

2. **Age Effect** is significant and negative
   - LinkedIn usage peaks in middle age groups (30-55)
   - Older demographics (65+) are much less likely to use LinkedIn
   - Focus marketing efforts on working-age professionals

3. **Secondary Factors:**
   - Marital status, parental status, and gender show weaker but still relevant effects
   - These can be used for fine-tuning target segments

### Marketing Recommendations:
- **Primary Target:** College-educated professionals, ages 30-55, with middle to high incomes
- **Platform Strategy:** LinkedIn is ideal for B2B marketing and professional services
- **Segmentation:** Create different campaigns for young professionals vs. established executives
- **Avoid:** Heavy investment in LinkedIn marketing for products targeting seniors or low-income segments

## Save the Model for Streamlit App

In [ ]:
# Save the trained model for use in Streamlit app
import pickle

with open('linkedin_model.pkl', 'wb') as file:
    pickle.dump(lr_model, file)

print("Model saved as 'linkedin_model.pkl'")
print("This model can now be loaded in the Streamlit app for real-time predictions.")